# 1. Profile the bag to get metadata + topics

In [1]:
from pathlib import Path
from hephaes_core.profiler import Profiler

bag_files = [
    'input/ros1.bag',
    # 'input/ros2.mcap',
    # 'input/sample.mcap',
    # 'input/sample2.mcap',
    ]
topics = []

if not bag_files:
    print("No bag files found in", input_dir)
else:
    prof = Profiler(bag_files, max_workers=1)
    metadata_list = prof.profile()
    for meta in metadata_list:
        print(f"File: {meta.file_path}")
        print(f"Path: {meta.path}")
        print(f"ROS version: {meta.ros_version}")
        print(f"Storage format: {meta.storage_format}")
        print(f"File size (bytes): {meta.file_size_bytes}")
        print(f"Start: {meta.start_time_iso} End: {meta.end_time_iso} Duration(s): {meta.duration_seconds}")
        print(f"Message count: {meta.message_count}")
        print("Topics:")
        topics = meta.topics
        for t in meta.topics:
            print(f"  - {t.name} ({t.message_type}): {t.message_count} messages, {t.rate_hz} Hz")
        print("-" * 40)




File: input/ros1.bag
Path: input/ros1.bag
ROS version: ROS1
Storage format: bag
File size (bytes): 37809346
Start: 2014-07-11T10:58:16.451314Z End: 2014-07-11T11:00:46.356676Z Duration(s): 149.9053622
Message count: 40671
Topics:
  - horizontal_laser_2d (sensor_msgs/msg/MultiEchoLaserScan): 5522 messages, 36.84 Hz
  - imu (sensor_msgs/msg/Imu): 29590 messages, 197.39 Hz
  - vertical_laser_2d (sensor_msgs/msg/MultiEchoLaserScan): 5559 messages, 37.1 Hz
----------------------------------------


# 2. Create a MappingTemplate

In [2]:
from hephaes_core.mappers import build_mapping_template

mapping = build_mapping_template(topics)

# 3. Use Converter to convert

In [7]:
from hephaes_core.converter import Converter

converter = Converter(bag_files, mapping, "./output").convert()

# 4. Stream the first few rows of the output

In [8]:
from hephaes_core.parquet import stream_wide_parquet_rows

output_parquet = "./output/episode_0001.parquet"
n_rows = 5

for i, row in enumerate(stream_wide_parquet_rows(output_parquet)):
    print(row)
    if i + 1 >= n_rows:
        break


{'episode_id': 'episode_0001', 'bag_path': 'input/ros1.bag', 'ros_version': 'ROS1', 'timestamp_ns': 1405076296451313900, 'horizontal_laser_2d': None, 'imu': '{"__bytes__":true,"encoding":"base64","value":"AAAAAEjDv1PsgOYaCAAAAGltdV9saW5rAAAAgEzEoT8AAABg7vBWPwAAAEC/6+o/AAAAQHlD4b8AAAAAAADwvwAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAInCTvwAAAOAS65s/AAAAYIJblL8AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA4rriPwAAAICWq9S/AAAAoL+oI0AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA="}', 'vertical_laser_2d': None}
{'episode_id': 'episode_0001', 'bag_path': 'input/ros1.bag', 'ros_version': 'ROS1', 'timestamp_ns': 1405076296454802000, 'horizontal_laser_2d': None, 'imu': '{"__bytes__":true,"encoding":"base64","value":"AAAAAEjDv1NQuhsbCAAAAGltdV9saW5rAAAAIBC9oT8AAACAUa9VPwAAAOCT6+o/AAAAwMRD4b8AAAAAAADwvwAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA